# EXP-028 v2 — признаки оценки закрытой части ответа

Проверяю вариант Г для критика PPO. Начинаю с готового разогрева сида 0 и замера стоимости расчёта. Для каждого действия беру сведения, доступные перед его токеном. Последний открытый слайд включаю в оценку после появления следующей завершённой строки `#SLIDE`.

Здесь измеряю извлечение признаков для будущей Z-lin. Чтобы сравнить качество голов, нужен следующий этап EXP-028 v2 на отложенных темах.

## 1. Настройки и данные

Читаю сохранённые токены, маски и режимы из `episodes.jsonl`. Беру по 16 эпизодов с поиском и без него. Выборка воспроизводима. Токенизатор и лимиты беру из того же конфига, что и в PPO. Для замера мне нужна память под bge-m3 на GPU 1.

In [ ]:
import random
import statistics
import sys
import time
from pathlib import Path

ROOT = Path.cwd().resolve()
if ROOT.name == "notebooks":
    ROOT = ROOT.parent
sys.path.insert(0, str(ROOT / "src"))

from presentation_rl.tasks import load_tasks
from presentation_rl.training.common import RewardModel, mode_task
from presentation_rl.training.ppo import PPOConfig, load_warmup_episodes
from presentation_rl.training.prefix_features import FEATURE_NAMES, PrefixFeatureExtractor

SEED = 0
PER_MODE = 16  # 10–16 на режим для первого замера
WARMUP = ROOT / "runs" / "ppo" / f"warmup_seed{SEED}"
config = PPOConfig.load(ROOT / "configs" / "algorithms" / "ppo.yaml", seed=SEED, warmup_dir=str(WARMUP))
episodes = load_warmup_episodes(WARMUP)
tasks = {task.id: task for task in load_tasks(ROOT / "data" / "tasks" / "tasks.json")}

rng = random.Random(SEED)
sample = []
for mode in ("plain", "search"):
    pool = [episode for episode in episodes if episode["mode"] == mode and any(episode["mask"])]
    if len(pool) < PER_MODE:
        raise ValueError(f"{mode}: в {WARMUP} только {len(pool)} доступных эпизодов, нужно {PER_MODE}")
    sample.extend(rng.sample(pool, PER_MODE))
rng.shuffle(sample)
print(f"Разогрев: {len(episodes)} эпизодов. Для замера: {len(sample)} ({PER_MODE} на режим).")

## 2. Загрузка токенизатора и оценки

Загружаю токенизатор исходной T-lite и bge-m3. Для каждой закрытой части собираю презентацию в памяти и считаю компоненты принятой оценки. Время загрузки модели записываю отдельно, потому что на сервере оно платится один раз за запуск.

In [ ]:
from transformers import AutoTokenizer

began = time.perf_counter()
tokenizer = AutoTokenizer.from_pretrained(config.policy_model, revision=config.policy_revision)
reward = RewardModel(ROOT, device=config.embedder_device, floor=config.reward_floor)
extract = PrefixFeatureExtractor(
    tokenizer, reward, max_new_tokens=config.max_new_tokens, max_context=config.max_context,
)
print(f"Загрузка: {(time.perf_counter() - began):.1f} с | эмбеддер: {config.embedder_device}")
print(f"Признаков: {len(FEATURE_NAMES)}")

## 3. Время расчёта признаков

Измеряю полное время извлечения на эпизод, включая декодирование, сборку префиксов и оценку. По среднему каждого режима оцениваю время пакета PPO. Показываю максимум, чтобы видеть дорогие эпизоды. Кеш эмбеддингов заполняется по ходу замера, а в основном PPO часть кеша уже заполнит расчёт итоговой награды.

Пересчитываю оценку на два запуска по 150 шагов. Ориентир 100 секунд на пакет взят из плана с 12 часами резерва. Его уточню по пилоту и фактическому времени старта.

In [ ]:
rows = []
for i, episode in enumerate(sample, start=1):
    mode = episode["mode"]
    task = mode_task(tasks[episode["task"]], mode)
    began = time.perf_counter()
    features = extract(task, episode["ids"], episode["mask"])
    seconds = time.perf_counter() - began
    rows.append({"mode": mode, "seconds": seconds, "scores": features.score_calls})
    print(f"{i:02d}/{len(sample)} · {mode} · {seconds:.2f} с · "
          f"{features.score_calls} оценок · {len(features.values)} действий", flush=True)

per_mode_step = config.tasks_per_mode * config.group_size
step_seconds = 0.0
step_scores = 0.0
for mode in ("plain", "search"):
    subset = [row for row in rows if row["mode"] == mode]
    times = [row["seconds"] for row in subset]
    average = statistics.fmean(times)
    step_seconds += per_mode_step * average
    step_scores += per_mode_step * statistics.fmean(row["scores"] for row in subset)
    print(f"{mode}: среднее {average:.2f} с, медиана {statistics.median(times):.2f} с, "
          f"максимум {max(times):.2f} с")

print(f"Добавка на шаг ({2 * per_mode_step} эпизода): {step_seconds:.1f} с, около {step_scores:.0f} оценок")
print(f"Добавка на два сида по {config.steps} шагов: {2 * config.steps * step_seconds / 3600:.2f} ч")